# Train the deployed model on Colab (RoBERTa-base, GoEmotions)

Runtime → Change runtime type → **GPU** (T4 or better). A T4 trains this in roughly 10–15 min versus ~25 min on a laptop RTX 3060.

Output: `model-v1.zip` containing `model.safetensors`, tokenizer, `thresholds.json`, metrics. Unzip it into `results/ml-artifacts/model-v1/` in the repo, then restart the backend.

In [ ]:
import torch; assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(torch.cuda.get_device_name(0))

In [ ]:
REPO = 'https://github.com/psoma10/KLH-CSIT-Y24-2420090069-Transformer-Based-Emotion-Aware-Customer-Feedback-Analysis.git'
# If the repo is private, use a token: https://<TOKEN>@github.com/psoma10/...
!git clone --depth 1 {REPO} repo
%cd repo/src/ml
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 scikit-learn==1.5.1 pyyaml==6.0.2

In [ ]:
# Builds data/goemotions_{train,validation,test}.jsonl (downloads GoEmotions from the HF hub)
!python data_prep.py --goemotions --out data/

In [ ]:
# Same hyper-parameters as configs/train_config.yaml; output goes to artifacts/model-v1
!python train_roberta.py --config configs/train_config.yaml

In [ ]:
# Tunes per-label thresholds on validation, scores the test split
!python evaluate.py --config configs/train_config.yaml

In [ ]:
!rm -rf artifacts/model-v1/checkpoints
!cd artifacts && zip -r ../model-v1.zip model-v1 && ls -lh ../model-v1.zip

In [ ]:
from google.colab import files
files.download('model-v1.zip')  # or copy to Drive:
# from google.colab import drive; drive.mount('/content/drive'); !cp model-v1.zip /content/drive/MyDrive/